# 03 · EDA Part B + ML Direction: NYC Yellow Taxi
**Owner:** Member 3 (EDA Lead B + ML Lead)

**Input:** the clean dataset from `01_cleaning.ipynb` (`data/processed/yellow_taxi_clean*.parquet`, 11.5M rows, the random 25% sample of all four files).

**Output:**
- `data/tableau/*.csv`: small aggregated tables that the Tableau visuals 4, 5 and 6 are built from
- `data/processed/eda_part2_summary.json`: every number quoted on the slides
- `ml_direction.md`: the proposed ML task, high-fare definition and justification from EDA

**Why aggregate here and draw in Tableau?** The GPU (RAPIDS cuDF) does the heavy work on 11.5M rows: filtering, area assignment and group-bys. Tableau then reads tables of a few hundred rows, so it stays fast and every member's numbers come from the same file.

| Visual | Question it answers | Tableau file |
|---|---|---|
| 4 | How does tipping change by hour, on weekdays and weekends? (credit-card trips only) | `tip_by_hour.csv` |
| 5 | Where do high-fare trips start? | `high_fare_by_area.csv` |
| 6 | How did January 2016 compare with January 2015 in trip volume and fare? | `daily_jan_compare.csv` |

## 1. GPU setup (RAPIDS cuDF)
On Colab: **Runtime → Change runtime type → T4 GPU**, then run this cell **first**, before pandas is imported anywhere.
`cudf.pandas` runs ordinary pandas code on the GPU and quietly falls back to the CPU for anything cuDF does not support. Without a GPU the notebook still runs on plain pandas, only slower.

In [ ]:
try:
    get_ipython().run_line_magic("load_ext", "cudf.pandas")
    BACKEND = "cuDF via cudf.pandas (GPU, RAPIDS)"
except Exception as e:
    BACKEND = "pandas (CPU)"
    print("cudf.pandas not available:", e)
print("Backend:", BACKEND)

### Colab only: bring in the clean data from Google Drive
Each Colab notebook runs on its own machine, so the parquet made by `01_cleaning.ipynb` has to come through Drive.
Copy `yellow_taxi_clean.parquet` and `cleaning_summary.json` into **My Drive → nyc-taxi** first. Outside Colab, this cell does nothing.

In [ ]:
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/nyc-taxi"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("data/processed", exist_ok=True)
    found = [f for f in os.listdir(DRIVE_DIR) if f.startswith("yellow_taxi_clean") or f == "cleaning_summary.json"]
    for f in found:
        if not os.path.exists(os.path.join("data/processed", f)):
            shutil.copy(os.path.join(DRIVE_DIR, f), os.path.join("data/processed", f))
    print("Copied from Drive:", found or "nothing found, check DRIVE_DIR")
except ImportError:
    print("Not on Colab, skipping Drive copy.")
except FileNotFoundError:
    print(f"Folder {DRIVE_DIR} not found. Create it and copy the parquet there.")

In [ ]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.path import Path as MplPath

warnings.filterwarnings("ignore")
T0 = time.time()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR, TAB_DIR = ROOT / "data" / "processed", ROOT / "data" / "tableau"
TAB_DIR.mkdir(parents=True, exist_ok=True)

HIGH_FARE_QUANTILE = 0.80      # "high-fare" = fare above the 80th percentile (top 20%). One setting, easy to change.
TIP_CLIP_QUANTILE = 0.99       # clip extreme tip % before taking means (medians are not affected)

# Sampling fraction used by Member 1 (to label trip counts correctly in Tableau)
try:
    KEEP_FRACTION = json.loads((PROC_DIR / "cleaning_summary.json").read_text())["keep_fraction"]
except Exception:
    KEEP_FRACTION = 0.25
print("Rows in the clean data are a", f"{KEEP_FRACTION:.0%}", "random sample of all trips")

## 2. Load the clean data
Accepts either file name (`yellow_taxi_clean.parquet` or `yellow_taxi_clean_1.parquet`), and Member 1's CSV fallback. Only the columns we need are loaded, which keeps GPU memory low.

In [ ]:
COLS = ["tpep_pickup_datetime", "pickup_longitude", "pickup_latitude", "fare_amount",
        "payment_type", "ratecodeid", "pickup_hour", "pickup_dow", "pickup_month", "is_weekend", "tip_pct"]

candidates = sorted(PROC_DIR.glob("yellow_taxi_clean*.parquet")) or sorted(PROC_DIR.glob("yellow_taxi_clean*.csv.gz"))
candidates = [p for p in candidates if "sample" not in p.name]
assert candidates, f"No clean data in {PROC_DIR}. Run 01_cleaning.ipynb first (or copy Member 1's parquet here)."
DATA_FILE = candidates[0]
print("Reading", DATA_FILE.name)

t = time.time()
if DATA_FILE.suffix == ".parquet":
    df = pd.read_parquet(DATA_FILE, columns=COLS)
else:
    df = pd.read_csv(DATA_FILE, usecols=COLS, parse_dates=["tpep_pickup_datetime"])
LOAD_SEC = time.time() - t
print(f"{len(df):,} rows x {df.shape[1]} columns loaded in {LOAD_SEC:.1f} s")

## 3. Define "high-fare" (the ML target)
The fare distribution is right-skewed, so we take a percentile rather than a fixed dollar amount: a trip is **high-fare** if its fare is above the 80th percentile. About one trip in five is positive, so the classes are **imbalanced**.

In [ ]:
t = time.time()
fare_q = df["fare_amount"].quantile([0.5, 0.75, HIGH_FARE_QUANTILE, 0.9, 0.99])
HIGH_FARE_THRESHOLD = float(fare_q.loc[HIGH_FARE_QUANTILE])
df["is_high_fare"] = (df["fare_amount"] > HIGH_FARE_THRESHOLD).astype("int8")
HIGH_FARE_SHARE = float(df["is_high_fare"].mean())

print("Fare percentiles ($):"); print(fare_q.round(2).to_string())
print(f"\nHigh-fare threshold: fare > ${HIGH_FARE_THRESHOLD:.2f}")
print(f"Positive class share: {HIGH_FARE_SHARE:.1%}  (imbalance ratio about 1 : {(1 - HIGH_FARE_SHARE) / HIGH_FARE_SHARE:.1f})")

## 4. Pickup areas from GPS
The data has GPS points but no taxi-zone IDs, so we assign **approximate** areas: boxes around the two airports, a polygon outline for Manhattan (split at 14th St and 59th St), and simple boundaries for the other boroughs.
These are approximations (borough edges are irregular). They are good enough to compare areas, and are listed as a limitation on the slides.

In [ ]:
t = time.time()
lon = df["pickup_longitude"].to_numpy(dtype="float64")
lat = df["pickup_latitude"].to_numpy(dtype="float64")

def box(lat_lo, lat_hi, lon_lo, lon_hi):
    return (lat >= lat_lo) & (lat <= lat_hi) & (lon >= lon_lo) & (lon <= lon_hi)

# Rough outline of Manhattan (lon, lat), west side going south, then east side going north
MANHATTAN = MplPath([
    (-73.927, 40.878), (-73.948, 40.850), (-73.962, 40.820), (-73.977, 40.800), (-73.988, 40.782),
    (-73.995, 40.772), (-74.008, 40.755), (-74.011, 40.741), (-74.017, 40.712), (-74.020, 40.700),
    (-74.010, 40.701), (-73.998, 40.707), (-73.975, 40.711), (-73.971, 40.727), (-73.971, 40.745),
    (-73.959, 40.759), (-73.943, 40.775), (-73.930, 40.795), (-73.934, 40.812), (-73.930, 40.835),
    (-73.910, 40.872), (-73.927, 40.878)])

in_man = np.zeros(len(df), dtype=bool)
cand = box(40.698, 40.880, -74.021, -73.907)          # only test points near Manhattan (much faster)
in_man[cand] = MANHATTAN.contains_points(np.column_stack([lon[cand], lat[cand]]))

jfk = box(40.620, 40.665, -73.835, -73.745)
lga = box(40.763, 40.783, -73.890, -73.855)
conditions = [
    jfk, lga,
    in_man & (lat < 40.737),
    in_man & (lat < 40.768),
    in_man,
    (lat >= 40.795) & (lon > -73.935),                               # Bronx (north/east of the Harlem River)
    (lat < 40.737) & (lon >= -74.040) & (lon < -73.855),             # Brooklyn
    (lat < 40.800) & (lon >= -73.960),                               # Queens
]
AREAS = ["JFK Airport", "LaGuardia Airport", "Manhattan: below 14th St", "Manhattan: 14th to 59th St",
         "Manhattan: above 59th St", "Bronx", "Brooklyn", "Queens"]
df["pickup_area"] = np.select(conditions, AREAS, default="Other (Staten Is., NJ, edges)")
del lon, lat, in_man, cand

print(f"Areas assigned in {time.time() - t:.1f} s")
print((df["pickup_area"].value_counts(normalize=True) * 100).round(2).to_string())

## 5. Visual 4 data: tip % by hour (credit-card trips only)
Cash tips are not recorded (see `challenges.md`), so only `payment_type == 1` is used. We keep the **median** (robust to extreme tips), a clipped mean, and the share of card trips with **no tip**.

In [ ]:
t = time.time()
cc = df[df["payment_type"] == 1][["pickup_hour", "is_weekend", "tip_pct"]].dropna()
clip_at = float(cc["tip_pct"].quantile(TIP_CLIP_QUANTILE))
cc["tip_pct_clipped"] = cc["tip_pct"].clip(upper=clip_at)
cc["no_tip"] = (cc["tip_pct"] == 0).astype("int8")

tip_by_hour = (cc.groupby(["pickup_hour", "is_weekend"])
                 .agg(trips=("tip_pct", "size"), median_tip_pct=("tip_pct", "median"),
                      mean_tip_pct_clipped=("tip_pct_clipped", "mean"), share_no_tip=("no_tip", "mean"))
                 .reset_index().sort_values(["is_weekend", "pickup_hour"]))
tip_by_hour = pd.DataFrame(tip_by_hour)            # make sure it is a plain pandas frame before export
tip_by_hour["day_type"] = np.where(tip_by_hour["is_weekend"] == 1, "Weekend", "Weekday")
tip_by_hour["share_no_tip"] = (tip_by_hour["share_no_tip"] * 100).round(2)
tip_by_hour[["median_tip_pct", "mean_tip_pct_clipped"]] = tip_by_hour[["median_tip_pct", "mean_tip_pct_clipped"]].round(2)
tip_by_hour.to_csv(TAB_DIR / "tip_by_hour.csv", index=False)

CC_SHARE = len(cc) / len(df)
OVERALL_MEDIAN_TIP = float(cc["tip_pct"].median())
wk = tip_by_hour[tip_by_hour["is_weekend"] == 0].set_index("pickup_hour")["mean_tip_pct_clipped"]
V4 = {"credit_card_share_of_trips_pct": round(CC_SHARE * 100, 1),
      "overall_median_tip_pct": round(OVERALL_MEDIAN_TIP, 2),
      "tip_clip_at_pct": round(clip_at, 1),
      "weekday_highest_mean_tip_hour": int(wk.idxmax()), "weekday_highest_mean_tip_pct": round(float(wk.max()), 2),
      "weekday_lowest_mean_tip_hour": int(wk.idxmin()), "weekday_lowest_mean_tip_pct": round(float(wk.min()), 2),
      "overall_share_no_tip_pct": round(float(cc["no_tip"].mean()) * 100, 1)}
del cc
print(f"Built in {time.time() - t:.1f} s"); print(json.dumps(V4, indent=2))
tip_by_hour.head()

**Insight (fill in after viewing the Tableau chart):** *e.g. "Card tips stay near X% all day but dip/peak at H:00; weekends differ by Y points."*

**Decision:** *e.g. "Hour changes tipping, so keep `pickup_hour` as a feature; tip analysis stays restricted to card trips."*

## 6. Visual 5 data: share of high-fare trips by pickup area
Counts are exported (not percentages) so Tableau can compute the share correctly with `SUM(high_fare_trips) / SUM(trips)`.

In [ ]:
t = time.time()
area = (df.groupby("pickup_area")
          .agg(trips=("is_high_fare", "size"), high_fare_trips=("is_high_fare", "sum"), sum_fare=("fare_amount", "sum"))
          .reset_index())
area = pd.DataFrame(area)
area["high_fare_share_pct"] = (area["high_fare_trips"] / area["trips"] * 100).round(2)
area["share_of_all_trips_pct"] = (area["trips"] / area["trips"].sum() * 100).round(2)
area["avg_fare"] = (area["sum_fare"] / area["trips"]).round(2)
area = area.sort_values("high_fare_share_pct", ascending=False)
area.to_csv(TAB_DIR / "high_fare_by_area.csv", index=False)

# Supporting numbers for the ML slide: does the label depend on things known at pickup?
by_hour = df.groupby("pickup_hour")["is_high_fare"].mean() * 100
by_rate = df.groupby("ratecodeid").agg(trips=("is_high_fare", "size"), high_share=("is_high_fare", "mean"))
by_rate["high_share"] = (by_rate["high_share"] * 100).round(1)
airport = df["pickup_area"].isin(["JFK Airport", "LaGuardia Airport"])

V5 = {"high_fare_threshold_usd": round(HIGH_FARE_THRESHOLD, 2),
      "high_fare_share_pct": round(HIGH_FARE_SHARE * 100, 1),
      "airport_share_of_trips_pct": round(float(airport.mean()) * 100, 2),
      "airport_share_of_high_fare_trips_pct": round(float(df.loc[df["is_high_fare"] == 1, "pickup_area"].isin(["JFK Airport", "LaGuardia Airport"]).mean()) * 100, 1),
      "high_fare_share_by_area_pct": dict(zip(area["pickup_area"], area["high_fare_share_pct"].astype(float))),
      "high_fare_share_by_hour_min": {"hour": int(by_hour.idxmin()), "pct": round(float(by_hour.min()), 1)},
      "high_fare_share_by_hour_max": {"hour": int(by_hour.idxmax()), "pct": round(float(by_hour.max()), 1)},
      "high_fare_share_by_ratecode_pct": {str(int(k)): float(v) for k, v in pd.DataFrame(by_rate)["high_share"].items()},
      "trips_by_ratecode": {str(int(k)): int(v) for k, v in pd.DataFrame(by_rate)["trips"].items()}}
print(f"Built in {time.time() - t:.1f} s"); print(json.dumps(V5, indent=2))
area

**Insight:** *e.g. "Only ~X% of all pickups are at the airports, but Y% of airport pickups are high-fare against Z% in Midtown."*

**Decision:** *e.g. "Pickup area is the strongest pickup-time signal, so the model gets a pickup-area feature; airport trips are kept, not treated as outliers."*

## 7. Visual 6 data: January 2015 vs January 2016
One row per day of January in each year. `trips` is the count in our 25% sample; `est_all_trips` scales it back up. The weekday is included because 1 January was a Thursday in 2015 and a Friday in 2016, so day-by-day lines are shifted by one weekday.

In [ ]:
t = time.time()
jan = df[df["pickup_month"] == 1][["tpep_pickup_datetime", "fare_amount", "pickup_dow"]]
jan = jan.assign(year=jan["tpep_pickup_datetime"].dt.year, day_of_month=jan["tpep_pickup_datetime"].dt.day)
daily = (jan.groupby(["year", "day_of_month"])
            .agg(trips=("fare_amount", "size"), sum_fare=("fare_amount", "sum"), pickup_dow=("pickup_dow", "first"))
            .reset_index())
daily = pd.DataFrame(daily).sort_values(["year", "day_of_month"])
daily["date"] = pd.to_datetime(dict(year=daily["year"], month=1, day=daily["day_of_month"]))
daily["weekday"] = daily["date"].dt.day_name()
daily["date"] = daily["date"].dt.strftime("%Y-%m-%d")      # plain text date, safe for Tableau
daily["avg_fare"] = (daily["sum_fare"] / daily["trips"]).round(2)
daily["est_all_trips"] = (daily["trips"] / KEEP_FRACTION).round().astype("int64")
daily.to_csv(TAB_DIR / "daily_jan_compare.csv", index=False)

yr = daily.groupby("year").agg(trips=("trips", "sum"), sum_fare=("sum_fare", "sum"), days=("trips", "size"))
yr["trips_per_day"] = yr["trips"] / yr["days"]
yr["avg_fare"] = yr["sum_fare"] / yr["trips"]
lowest = daily.loc[daily.groupby("year")["trips"].idxmin()]
med_by_year = daily.groupby("year")["trips"].median()
V6 = {"jan_trips_sample": {int(k): int(v) for k, v in yr["trips"].items()},
      "jan_avg_fare_usd": {int(k): round(float(v), 2) for k, v in yr["avg_fare"].items()}}
if {2015, 2016} <= set(yr.index):
    V6["trip_volume_change_pct"] = round(float(yr.loc[2016, "trips_per_day"] / yr.loc[2015, "trips_per_day"] - 1) * 100, 1)
    V6["avg_fare_change_pct"] = round(float(yr.loc[2016, "avg_fare"] / yr.loc[2015, "avg_fare"] - 1) * 100, 1)
V6["lowest_day"] = {int(r.year): {"date": str(r.date), "weekday": r.weekday,
                                  "trips_vs_year_median_pct": round(float(r.trips / med_by_year[r.year] - 1) * 100, 1)}
                    for r in lowest.itertuples()}
del jan
print(f"Built in {time.time() - t:.1f} s"); print(json.dumps(V6, indent=2))
daily.head()

**Insight:** *e.g. "January 2016 had X% fewer trips per day than January 2015 while the average fare changed by Y%; one day in late January 2016 collapses (check the date)."*

**Decision:** *e.g. "Year/month must be a model feature, and single-day shocks are kept but noted; volume falls between years, so do not pool years blindly."*

## 8. Save the numbers for the slides and draft `ml_direction.md`

In [ ]:
TOTAL_SEC = round(time.time() - T0, 1)
summary = {"backend": BACKEND, "data_file": DATA_FILE.name, "rows": int(len(df)), "keep_fraction": KEEP_FRACTION,
           "load_sec": round(LOAD_SEC, 1), "total_runtime_sec": TOTAL_SEC,
           "visual_4_tips": V4, "visual_5_high_fare_area": V5, "visual_6_jan_compare": V6}
(PROC_DIR / "eda_part2_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(f"Saved eda_part2_summary.json. Total runtime {TOTAL_SEC} s on {BACKEND}")
for f in sorted(TAB_DIR.glob("*.csv")):
    print(f"  {f.relative_to(ROOT)}  ({sum(1 for _ in open(f)) - 1} rows)")

In [ ]:
rate_lines = "\n".join(f"| {k} | {V5['trips_by_ratecode'][k]:,} | {v}% |" for k, v in V5["high_fare_share_by_ratecode_pct"].items())
area_lines = "\n".join(f"| {a} | {s}% |" for a, s in V5["high_fare_share_by_area_pct"].items())
vol = V6.get("trip_volume_change_pct", "n/a")

text = f'''# ML direction (Member 3)

> Mid-presentation: **direction and justification only, no model results.**
> Numbers below come from `notebooks/03_eda_part2.ipynb` ({summary["rows"]:,} clean trips, a {KEEP_FRACTION:.0%} random sample).

## Proposed task
**Binary classification:** predict whether a trip will be **high-fare**, using only information known **at pickup**.

## Definition of "high-fare"
A trip is high-fare if `fare_amount` is above the **{int(HIGH_FARE_QUANTILE * 100)}th percentile = ${V5["high_fare_threshold_usd"]:.2f}**.
- Why a percentile: fares are strongly right-skewed (long airport trips), so a fixed round number would be arbitrary; the percentile adapts to the data and is easy to explain.
- Result: **{V5["high_fare_share_pct"]}%** of trips are positive, so the classes are **imbalanced** (about 1 : {(100 - V5["high_fare_share_pct"]) / V5["high_fare_share_pct"]:.1f}).

## Why classification, and why with resampling (from the EDA)
1. **The target is rare** ({V5["high_fare_share_pct"]}% positive). A model that always says "not high-fare" would be {100 - V5["high_fare_share_pct"]:.0f}% accurate and useless, so we will use **class weights or SMOTE** and judge models by **F1 / PR-AUC**, not accuracy.
2. **Pickup area separates the classes (Visual 5).** Airports are {V5["airport_share_of_trips_pct"]}% of all pickups but {V5["airport_share_of_high_fare_trips_pct"]}% of high-fare trips. High-fare share by area:

| Pickup area | High-fare share |
|---|---:|
{area_lines}

3. **Time matters (Visuals 4 and 6).** High-fare share moves from {V5["high_fare_share_by_hour_min"]["pct"]}% at {V5["high_fare_share_by_hour_min"]["hour"]}:00 to {V5["high_fare_share_by_hour_max"]["pct"]}% at {V5["high_fare_share_by_hour_max"]["hour"]}:00, and January 2016 trip volume changed by {vol}% per day against January 2015, so hour, weekday and month/year are features.
4. Why not regression: predicting the exact fare at pickup is much harder without the distance, and the business question ("is this going to be an expensive trip?") is a yes/no. Clustering is kept as an optional extra (grouping pickup areas).

## Features (known at pickup) and leakage rules
**Use:** `pickup_hour`, `pickup_dow`, `is_weekend`, `pickup_month` / year, `pickup_area` (or pickup lat/long), `passenger_count`, `vendorid`.
**Do not use** (only known after the trip): `trip_distance`, `duration_min`, `speed_mph`, `dropoff_*`, `tip_amount`, `total_amount`, `extra`, `mta_tax`, `tolls_amount`.

**Open decision, rate code.** `RatecodeID` is set when the trip starts, but code 2 is the JFK flat fare, so it nearly gives the answer away:

| Rate code | Trips | High-fare share |
|---|---:|---:|
{rate_lines}

Plan: train **with and without** rate code and report both, so the model's skill is not just "it is a JFK flat-fare trip".

## Validation plan (final presentation)
Stratified split (or train on 2015 + early 2016, test on March 2016), baseline = logistic regression with class weights, then a tree model (random forest / gradient boosting); compare SMOTE vs class weights on PR-AUC and F1.
'''
out = ROOT / "ml_direction.md"
if out.exists():
    out = ROOT / "ml_direction_autogen.md"      # never overwrite a hand-edited version
out.write_text(text)
print("Wrote", out.name)